# Score AlphaGenome outputs for Supplementary Figure S5

**Purpose.** Generate the splice-specific AlphaGenome output table used by Supplementary Figure S5.

## Reproducibility contract

- **Run from:** the repository root or this notebook's model directory.
- **Genome build:** GRCh38/hg38.
- **Variant key:** `#CHROM`, `POS`, `REF`, and `ALT`.
- **Input:** `data/sample_data.csv.gz`.
- **Output:** `data/model_scores/alphagenome_all_outputs.csv.gz`.
- **Source:** https://deepmind.google.com/science/alphagenome/
- **Prepare input:** Set `ALPHAGENOME_API_KEY` in the environment.


## 1. Setup


In [ ]:
import os
from pathlib import Path

# Support kernels started from either the repository root or this notebook's directory.
START_DIR = Path.cwd().resolve()
REPO_ROOT = START_DIR if (START_DIR / "data" / "sample_data.csv.gz").is_file() else START_DIR.parent
DATA_DIR = REPO_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
MODEL_SCORE_DIR = DATA_DIR / "model_scores"
BENCHMARK_PATH = DATA_DIR / "sample_data.csv.gz"
VARIANT_KEY = ["#CHROM", "POS", "REF", "ALT"]

MODEL_SCORE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH = MODEL_SCORE_DIR / "alphagenome_all_outputs.csv.gz"
CHECKPOINT_DIR = MODEL_SCORE_DIR / "alphagenome_checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)


## 2. Install dependencies and load inputs


## 3. Run scoring or score mapping


In [ ]:
from IPython.display import clear_output
%pip uninstall -y numpy
%pip install "numpy<2.0.0" --upgrade
%pip install alphagenome==0.6.1


## 4. Validate and save output


In [ ]:
import os
import time
import random
import pandas as pd
from alphagenome.data import genome
from alphagenome.models import dna_client, variant_scorers
from tqdm import tqdm
import numpy as np
from concurrent.futures import ThreadPoolExecutor, as_completed

# 0. API key
# Read the API key from the environment.
API_KEY = os.environ.get("ALPHAGENOME_API_KEY")
if not API_KEY:
    raise RuntimeError("Set ALPHAGENOME_API_KEY before running this notebook")

# Never paste an API key into this notebook.
dna_model = dna_client.create(API_KEY)

# 1. Configuration
CSV_PATH = BENCHMARK_PATH
ORGANISM_NAME = "human"
SEQUENCE_LENGTH = "1MB"

# Use ten workers while retaining retry and backoff handling for API requests.
MAX_WORKERS = 10

# Retry count for an individual variant
MAX_RETRIES = 3

# Number of completed variants between checkpoints
CHECKPOINT_EVERY = 100000

scorer_selections = {
    "rna_seq": True,
    "cage": True,
    "procap": True,
    "atac": True,
    "dnase": True,
    "chip_histone": True,
    "chip_tf": True,
    "polyadenylation": True,
    "splice_sites": True,
    "splice_site_usage": True,
    "splice_junctions": True,
}

# 2. Initialize organism and scorers
organism_map = {
    "human": dna_client.Organism.HOMO_SAPIENS,
    "mouse": dna_client.Organism.MUS_MUSCULUS,
}
organism = organism_map[ORGANISM_NAME]

sequence_length = dna_client.SUPPORTED_SEQUENCE_LENGTHS[
    f"SEQUENCE_LENGTH_{SEQUENCE_LENGTH}"
]

all_scorers = variant_scorers.RECOMMENDED_VARIANT_SCORERS

selected_scorers = [
    all_scorers[key]
    for key in all_scorers
    if scorer_selections.get(key.lower(), False)
]

unsupported_scorers = [
    scorer for scorer in selected_scorers
    if (
        organism.value not in variant_scorers.SUPPORTED_ORGANISMS[scorer.base_variant_scorer]
        or (
            scorer.requested_output == dna_client.OutputType.PROCAP
            and organism == dna_client.Organism.MUS_MUSCULUS
        )
    )
]

for scorer in unsupported_scorers:
    selected_scorers.remove(scorer)

print("Selected scorers:")
for s in selected_scorers:
    print(" -", s)

# 3. Load and preprocess input CSV
csv = pd.read_csv(CSV_PATH, dtype={"#CHROM": str}, low_memory=False)

csv["ClinVarName_splice"] = pd.to_numeric(
    csv["ClinVarName_splice"],
    errors="coerce"
)

csv = csv[csv["ClinVarName_splice"] == 1].copy()
print("Rows with ClinVarName_splice == 1:", len(csv))
display(csv.head())

print("Total rows in CSV before filtering:", len(csv))

# Preserve original variant keys for the canonical output.
output_keys = csv[VARIANT_KEY].copy()
output_keys["#CHROM"] = output_keys["#CHROM"].astype(str)

csv["#CHROM"] = csv["#CHROM"].astype(str)
csv["#CHROM"] = csv["#CHROM"].apply(
    lambda x: x if x.startswith("chr") else f"chr{x}"
)

csv["variant_id"] = (
    csv["#CHROM"].astype(str)
    + "_"
    + csv["POS"].astype(str)
    + "_"
    + csv["REF"].astype(str)
    + "_"
    + csv["ALT"].astype(str)
    + "_b38"
)
input_variant_ids = csv["variant_id"].copy()

valid_chroms = {f"chr{i}" for i in range(1, 23)} | {"chrX", "chrY", "chrM"}
csv = csv[csv["#CHROM"].isin(valid_chroms)].copy()

csv = csv[
    csv["REF"].astype(str).str.match("^[ACGT]$")
    & csv["ALT"].astype(str).str.match("^[ACGT]$")
].copy()

required_columns = ["variant_id", "#CHROM", "POS", "REF", "ALT"]
for col in required_columns:
    if col not in csv.columns:
        raise ValueError(f"Missing required column: {col}")

print("Total rows after filtering:", len(csv))

# 4. Helper functions
def convert_variant_id(v):
    """
    Convert:
        chr1:930165:G>A
    into:
        chr1_930165_G_A_b38
    """
    chrom, pos, change = str(v).split(":")
    ref, alt = change.split(">")
    return f"{chrom}_{pos}_{ref}_{alt}_b38"


def score_one_variant(item):
    """
    Score one variant.
    item is (row_index, row_dict).
    Returns:
        ("success", df_scores)
    or:
        ("failed", failed_record)
    """
    i, row = item

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            variant = genome.Variant(
                chromosome=str(row["#CHROM"]),
                position=int(row["POS"]),
                reference_bases=str(row["REF"]),
                alternate_bases=str(row["ALT"]),
                name=str(row["variant_id"]),
            )

            interval = variant.reference_interval.resize(sequence_length)

            variant_scores = dna_model.score_variant(
                interval=interval,
                variant=variant,
                variant_scorers=selected_scorers,
                organism=organism,
            )

            df_scores = variant_scorers.tidy_scores([variant_scores]).copy()

            df_scores["alphagenome_variant_id"] = df_scores["variant_id"].astype(str)
            df_scores["variant_id"] = df_scores["alphagenome_variant_id"].apply(
                convert_variant_id
            )
            df_scores["input_row_index"] = i

            return "success", df_scores

        except Exception as e:
            error_msg = str(e)

            # Exponential backoff for rate limits or transient network failures
            sleep_time = (2 ** (attempt - 1)) + random.uniform(0, 1)
            time.sleep(sleep_time)

            if attempt == MAX_RETRIES:
                failed_record = {
                    "row_index": i,
                    "variant_id": row.get("variant_id", None),
                    "#CHROM": row.get("#CHROM", None),
                    "POS": row.get("POS", None),
                    "REF": row.get("REF", None),
                    "ALT": row.get("ALT", None),
                    "error": error_msg,
                }
                return "failed", failed_record


def save_checkpoint(all_tidy_scores, failed_records, suffix="checkpoint"):
    """
    Save current partial results.
    """
    if len(all_tidy_scores) > 0:
        df_ckpt = pd.concat(all_tidy_scores, ignore_index=True)
        df_ckpt.to_csv(CHECKPOINT_DIR / f"tidy_all_scores_{suffix}.csv", index=False)

    if len(failed_records) > 0:
        df_failed_ckpt = pd.DataFrame(failed_records)
        df_failed_ckpt.to_csv(CHECKPOINT_DIR / f"failed_records_{suffix}.csv", index=False)


# 5. Parallel scoring
records = list(csv.iterrows())

all_tidy_scores = []
failed_records = []

print(f"Start parallel scoring with MAX_WORKERS={MAX_WORKERS}")
print(f"Total variants to score: {len(records)}")

completed = 0

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    future_to_index = {
        executor.submit(score_one_variant, (i, row)): i
        for i, row in records
    }

    for future in tqdm(
        as_completed(future_to_index),
        total=len(future_to_index),
        desc="Scoring variants in parallel"
    ):
        status, result = future.result()
        completed += 1

        if status == "success":
            all_tidy_scores.append(result)
        else:
            failed_records.append(result)

        if completed % CHECKPOINT_EVERY == 0:
            print(f"\nCheckpoint at {completed}/{len(records)}")
            print(f"Success chunks: {len(all_tidy_scores)}")
            print(f"Failed records: {len(failed_records)}")
            save_checkpoint(
                all_tidy_scores,
                failed_records,
                suffix=f"checkpoint_{completed}"
            )

# Save failed records
df_failed = pd.DataFrame(failed_records)
df_failed.to_csv(CHECKPOINT_DIR / "failed_records.csv", index=False)

if len(all_tidy_scores) == 0:
    raise RuntimeError("No variant was successfully scored.")

df_tidy_all = pd.concat(all_tidy_scores, ignore_index=True)

#df_tidy_all.to_csv("alphagenome_tidy_all_scores.csv", index=False)

print("Saved full tidy scores:")
print(df_tidy_all.shape)
print(df_tidy_all.columns.tolist())
display(df_tidy_all.head())

print("Failed records:", len(df_failed))
if len(df_failed) > 0:
    display(df_failed.head())

# 6. Build all-head wide table
df_tidy_all["quantile_abs"] = df_tidy_all["quantile_score"].abs()
df_tidy_all["raw_abs"] = df_tidy_all["raw_score"].abs()

q_wide = (
    df_tidy_all
    .groupby(["variant_id", "output_type"])["quantile_abs"]
    .max()
    .unstack("output_type")
)

r_wide = (
    df_tidy_all
    .groupby(["variant_id", "output_type"])["raw_abs"]
    .max()
    .unstack("output_type")
)

q_wide.columns = [f"{c}_quantile_abs" for c in q_wide.columns]
r_wide.columns = [f"{c}_raw_abs" for c in r_wide.columns]

df_heads = pd.concat([q_wide, r_wide], axis=1).reset_index()

df_heads = df_heads.rename(columns={
    "SPLICE_JUNCTIONS_quantile_abs": "SpliceJunction_quantile_abs",
    "SPLICE_JUNCTIONS_raw_abs": "SpliceJunction_raw_abs",
})

print("All-head wide table columns:")
print(df_heads.columns.tolist())
display(df_heads.head())

# 7. Construct composite scores
all_quantile_cols = [
    c for c in df_heads.columns
    if c.endswith("_quantile_abs")
]

df_heads["composite_quantile_score"] = df_heads[all_quantile_cols].max(axis=1)

all_raw_cols = [
    c for c in df_heads.columns
    if c.endswith("_raw_abs")
]

df_heads["composite_raw_score"] = df_heads[all_raw_cols].max(axis=1)

splice_quantile_cols = [
    c for c in [
        "SPLICE_SITES_quantile_abs",
        "SPLICE_SITE_USAGE_quantile_abs",
        "SpliceJunction_quantile_abs",
    ]
    if c in df_heads.columns
]

splice_raw_cols = [
    c for c in [
        "SPLICE_SITES_raw_abs",
        "SPLICE_SITE_USAGE_raw_abs",
        "SpliceJunction_raw_abs",
    ]
    if c in df_heads.columns
]

print("splice_quantile_cols:", splice_quantile_cols)
print("splice_raw_cols:", splice_raw_cols)

df_heads["splice_head_composite_quantile_score"] = df_heads[
    splice_quantile_cols
].max(axis=1)

df_heads["aggregate_splice"] = df_heads[
    splice_raw_cols
].max(axis=1)

# 8. Map scores back to the original variant keys.
score_columns = [column for column in df_heads.columns if column != "variant_id"]
df_final = output_keys.copy()
df_final["variant_id"] = input_variant_ids
df_final = df_final.merge(
    df_heads, on="variant_id", how="left", validate="many_to_one"
)
df_final = df_final[VARIANT_KEY + score_columns]

df_final.to_csv(OUTPUT_PATH, index=False, compression="gzip")

print(f"Final table saved: {OUTPUT_PATH}")
print(df_final.shape)
display(df_final.head())

# 9. Quick missingness check
old_analysis_cols = [
    "SPLICE_SITES_quantile_abs",
    "SPLICE_SITES_raw_abs",
    "SPLICE_SITE_USAGE_quantile_abs",
    "SPLICE_SITE_USAGE_raw_abs",
    "SpliceJunction_quantile_abs",
    "SpliceJunction_raw_abs",
    "composite_quantile_score",
    "splice_head_composite_quantile_score",
    "aggregate_splice",
]

existing_old_analysis_cols = [
    c for c in old_analysis_cols
    if c in df_final.columns
]

print("Missingness for old analysis columns:")
print(df_final[existing_old_analysis_cols].isna().sum())
